In [1]:
import pandas as pd, numpy as np

df = pd.read_csv(r"D:\MBFM_project\master_features.csv", index_col=0, parse_dates=True)

print("Shape:", df.shape)
print("\nNaN counts in HAR columns:")
cols = ["rv", "rv_lag1", "rv_5d", "rv_22d", "vix_level", "ret_t"]
print(df[cols].isna().sum())

print("\nFirst valid row index:")
print(df[cols].dropna().index[0])

print("\nRows 1495–1505 of rv_lag1:")
print(df["rv_lag1"].iloc[1495:1506])

print("\nRows 1495–1505 of rv:")
print(df["rv"].iloc[1495:1506])

Shape: (5199, 24)

NaN counts in HAR columns:
rv           0
rv_lag1      0
rv_5d        0
rv_22d       0
vix_level    0
ret_t        0
dtype: int64

First valid row index:
2005-04-04 00:00:00

Rows 1495–1505 of rv_lag1:
date
2011-06-02    0.046670
2011-06-03    0.045966
2011-06-06    0.046821
2011-06-07    0.044505
2011-06-08    0.044468
2011-06-09    0.042589
2011-06-10    0.043082
2011-06-13    0.043020
2011-06-14    0.044350
2011-06-15    0.044445
2011-06-16    0.043861
Name: rv_lag1, dtype: float64

Rows 1495–1505 of rv:
date
2011-06-02    0.045966
2011-06-03    0.046821
2011-06-06    0.044505
2011-06-07    0.044468
2011-06-08    0.042589
2011-06-09    0.043082
2011-06-10    0.043020
2011-06-13    0.044350
2011-06-14    0.044445
2011-06-15    0.043861
2011-06-16    0.045477
Name: rv, dtype: float64


In [2]:
print(df.columns.tolist())
print(df.iloc[1498:1502][["rv","rv_lag1","rv_5d","rv_22d"]].to_string())

['rv', 'rv_lag1', 'rv_5d', 'rv_22d', 'sq_return', 'vix_level', 'vix_chg', 'brent_ret', 'dxy_level', 'dxy_ret', 'india10y', 'india10y_chg', 'us10y', 'us10y_chg', 'rate_spread', 'callmoney', 'nifty_ret', 'em_ret', 'ret_t', 'is_mpc_day', 'is_fomc_day', 'day_of_week', 'month', 'week_of_year']
                  rv   rv_lag1     rv_5d    rv_22d
date                                              
2011-06-07  0.044468  0.044505  0.045768  0.055059
2011-06-08  0.042589  0.044468  0.045686  0.054014
2011-06-09  0.043082  0.042589  0.044870  0.052815
2011-06-10  0.043020  0.043082  0.044293  0.051654


In [3]:
import numpy as np, pandas as pd, os

FCAST  = r"D:\MBFM_project\results\forecasts"
DATA   = r"D:\MBFM_project\master_features.csv"
df     = pd.read_csv(DATA, index_col=0, parse_dates=True)
actual = df["rv"].values[1500:]
TRAIN  = 1500

files = {
    "GARCH(1,1)"    : "garch11.npy",
    "EGARCH"        : "egarch.npy",
    "GJR-GARCH"     : "gjr_garch.npy",
    "HAR-RV"        : "har_rv.npy",
    "HAR-RV-X"      : "har_rv_x.npy",
    "Random Forest" : "random_forest.npy",
    "XGBoost"       : "xgboost.npy",
}

print(f"{'Model':<16} {'OOS_n':>6} {'RMSE':>10} {'MAE':>10} {'QLIKE':>10}")
print("-" * 56)
for name, fname in files.items():
    path = os.path.join(FCAST, fname)
    if not os.path.exists(path):
        print(f"{name:<16} {'MISSING':>6}")
        continue
    fc = np.load(path)
    # if saved as full-length array, slice to OOS only
    if len(fc) == len(df):
        fc = fc[TRAIN:]
    mask = ~np.isnan(fc) & (fc > 0)
    if mask.sum() == 0:
        print(f"{name:<16} {'n=0':>6}")
        continue
    a, f_ = actual[mask], fc[mask]
    rmse  = np.sqrt(np.mean((f_ - a)**2))
    mae   = np.mean(np.abs(f_ - a))
    qlike = np.mean(a/f_ - np.log(a/f_) - 1)
    print(f"{name:<16} {mask.sum():>6} {rmse:>10.6f} {mae:>10.6f} {qlike:>10.6f}")

Model             OOS_n       RMSE        MAE      QLIKE
--------------------------------------------------------
GARCH(1,1)         3699   0.033610   0.025026   0.154288
EGARCH             3699   0.031356   0.023141   0.131604
GJR-GARCH          3699   0.035170   0.026186   0.158456
HAR-RV             3699   0.003470   0.001943   0.002243
HAR-RV-X           3699   0.003484   0.001982   0.002262
Random Forest      3699   0.009364   0.004084   0.010747
XGBoost            3699   0.006806   0.002752   0.005316


In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.memory_allocated(0))

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

True
NVIDIA GeForce RTX 4060 Laptop GPU
0
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.6 GB


In [3]:
import numpy as np

f = np.load(r"D:\MBFM_project\results\forecasts\lstm.npy")
a = np.load  # just read from csv instead:

import pandas as pd
df = pd.read_csv(r"D:\MBFM_project\master_features.csv", index_col=0)
actual = df["rv"].values[1500:].astype(float)

print("forecast min  :", f.min())
print("forecast max  :", f.max())
print("negatives     :", (f <= 0).sum())
print("near-zero<1e-4:", (f < 1e-4).sum())
print("actual min    :", actual.min())
print("actual max    :", actual.max())
print("\nworst QLIKE contributors:")
qlike_i = actual/f - np.log(actual/f) - 1
worst = np.argsort(qlike_i)[::-1][:10]
for idx in worst:
    print(f"  i={idx}  actual={actual[idx]:.6f}  forecast={f[idx]:.6f}  qlike={qlike_i[idx]:.2f}")

forecast min  : 1e-08
forecast max  : 0.20638707280158997
negatives     : 0
near-zero<1e-4: 1
actual min    : 0.0040830555196029
actual max    : 0.2806127784765368

worst QLIKE contributors:
  i=3173  actual=0.006198  forecast=0.000000  qlike=619736.65
  i=3174  actual=0.006496  forecast=0.002012  qlike=1.06
  i=3170  actual=0.005568  forecast=0.018816  qlike=0.51
  i=3168  actual=0.004083  forecast=0.011828  qlike=0.41
  i=3169  actual=0.005503  forecast=0.015903  qlike=0.41
  i=3171  actual=0.005591  forecast=0.013656  qlike=0.30
  i=3231  actual=0.004645  forecast=0.011287  qlike=0.30
  i=2993  actual=0.006991  forecast=0.016981  qlike=0.30
  i=2992  actual=0.007433  forecast=0.016916  qlike=0.26
  i=2994  actual=0.007192  forecast=0.016246  qlike=0.26
